## Part 1 – Regex-Based Text Cleaning in Python

In [1]:
import re
from pathlib import Path

In [2]:
data_path = Path("data")
data_path.mkdir(exist_ok=True)

raw_file = data_path / "tweets_raw.txt"

if not raw_file.exists():
    sample_lines = [
        "RT @user1: Check out https://example.com !!! #NLP #AI 😄",
        "Error 500 at 2026-02-11T10:23:45Z from 192.168.0.1 - see http://errors.example.org?id=500",
        "New blog post → https://myblog.org/post/123 #blogging #python",
        "@user2 lol that was wild 😂😂 http://bit.ly/xyz",
    ]
    raw_file.write_text("\n".join(sample_lines), encoding="utf-8")

raw_text = raw_file.read_text(encoding="utf-8")
print(raw_text.splitlines()[:3])

['RT @user1: Check out https://example.com !!! #NLP #AI 😄', 'Error 500 at 2026-02-11T10:23:45Z from 192.168.0.1 - see http://errors.example.org?id=500', 'New blog post → https://myblog.org/post/123 #blogging #python']


### Define Regex Patterns

In [3]:
# URLs: simple pattern (good enough for this lab)
URL_PATTERN = re.compile(r'https?://\S+')
# @mentions: @ followed by letters/digits/underscore
MENTION_PATTERN = re.compile(r'@\w+')
# hashtags: # followed by letters/digits/underscore
HASHTAG_PATTERN = re.compile(r'#\w+')
# emojis / non-ASCII: here we just detect non-ASCII as crude approximation
NON_ASCII_PATTERN = re.compile(r'[^\x00-\x7F]+')
# collapse multiple spaces
MULTISPACE_PATTERN = re.compile(r'\s+')

### Apply Cleaning Pipeline

In [6]:
def clean_line(line: str) -> str:
    line = URL_PATTERN.sub("<URL>", line)
    line = MENTION_PATTERN.sub("<USER>", line)
    line = HASHTAG_PATTERN.sub("<HASHTAG>", line)
    line = NON_ASCII_PATTERN.sub("<EMOJI>", line)
    line = line.lower()
    line = MULTISPACE_PATTERN.sub(" ", line)
    return line.strip()
cleaned_lines = [clean_line(ln) for ln in raw_text.splitlines()]

for original, cleaned in zip(raw_text.splitlines(), cleaned_lines):
    print("ORIG:", original)
    print("CLEAN:", cleaned)
    print("---")

ORIG: RT @user1: Check out https://example.com !!! #NLP #AI 😄
CLEAN: rt <user>: check out <url> !!! <hashtag> <hashtag> <emoji>
---
ORIG: Error 500 at 2026-02-11T10:23:45Z from 192.168.0.1 - see http://errors.example.org?id=500
CLEAN: error 500 at 2026-02-11t10:23:45z from 192.168.0.1 - see <url>
---
ORIG: New blog post → https://myblog.org/post/123 #blogging #python
CLEAN: new blog post <emoji> <url> <hashtag> <hashtag>
---
ORIG: @user2 lol that was wild 😂😂 http://bit.ly/xyz
CLEAN: <user> lol that was wild <emoji> <url>
---


### Save Cleaned Version

In [7]:
clean_file = data_path / "tweets_cleaned.txt"
clean_file.write_text("\n".join(cleaned_lines), encoding="utf-8")
print("Saved cleaned file:", clean_file)

Saved cleaned file: data\tweets_cleaned.txt


### Extract from Raw Text

In [8]:
lines = raw_text.splitlines()
all_urls = []
all_mentions = []
all_hashtags = []
for ln in lines:
 all_urls.extend(URL_PATTERN.findall(ln))
 all_mentions.extend(MENTION_PATTERN.findall(ln))
 all_hashtags.extend(HASHTAG_PATTERN.findall(ln))
print("URLs:", all_urls)
print("Mentions:", all_mentions)
print("Hashtags:", all_hashtags)

URLs: ['https://example.com', 'http://errors.example.org?id=500', 'https://myblog.org/post/123', 'http://bit.ly/xyz']
Mentions: ['@user1', '@user2']
Hashtags: ['#NLP', '#AI', '#blogging', '#python']


### Unique & Frequency Counts

In [9]:
from collections import Counter
print("Top URLs:", Counter(all_urls).most_common())
print("Top Mentions:", Counter(all_mentions).most_common())
print("Top Hashtags:", Counter(all_hashtags).most_common())

Top URLs: [('https://example.com', 1), ('http://errors.example.org?id=500', 1), ('https://myblog.org/post/123', 1), ('http://bit.ly/xyz', 1)]
Top Mentions: [('@user1', 1), ('@user2', 1)]
Top Hashtags: [('#NLP', 1), ('#AI', 1), ('#blogging', 1), ('#python', 1)]


In [10]:
IP_PATTERN = re.compile(r'\b(?:\d{1,3}\.){3}\d{1,3}\b')
ips = []
for ln in lines:
 ips.extend(IP_PATTERN.findall(ln))
print("IPs:", ips)

IPs: ['192.168.0.1']


## Part 2 – Subword Tokenization with HuggingFace (BPE)

### Install and Import

In [11]:
!pip install tokenizers
from tokenizers import Tokenizer
from tokenizers.models import BPE
from tokenizers.trainers import BpeTrainer
from tokenizers.pre_tokenizers import Whitespace
from tokenizers.processors import TemplateProcessing


[notice] A new release of pip is available: 26.0 -> 26.0.1
[notice] To update, run: python.exe -m pip install --upgrade pip


### Prepare a Toy Corpus

In [12]:
toy_corpus_file = data_path / "toy_corpus.txt"
if not toy_corpus_file.exists():
 extra_sentences = [
 "the water of walden pond is so beautifully clear",
 "low new newer newest lower lowest",
 "subword tokenization helps with rareword rarewords ultra-rareword",
 "renew reset renews renewing renewal",
 "byte pair encoding bpe learns merges for frequent pairs",
 "this corpus is tiny but good enough for demonstration",
 ]
 corpus_text = "\n".join(cleaned_lines + extra_sentences)
 toy_corpus_file.write_text(corpus_text, encoding="utf-8")
print(toy_corpus_file.read_text(encoding="utf-8").splitlines()[:10])

['rt <user>: check out <url> !!! <hashtag> <hashtag> <emoji>', 'error 500 at 2026-02-11t10:23:45z from 192.168.0.1 - see <url>', 'new blog post <emoji> <url> <hashtag> <hashtag>', '<user> lol that was wild <emoji> <url>', 'the water of walden pond is so beautifully clear', 'low new newer newest lower lowest', 'subword tokenization helps with rareword rarewords ultra-rareword', 'renew reset renews renewing renewal', 'byte pair encoding bpe learns merges for frequent pairs', 'this corpus is tiny but good enough for demonstration']


### Helper Function: Train BPE

In [13]:
def train_bpe_tokenizer(corpus_files, vocab_size=200, special_tokens=None):
    
    if special_tokens is None:
        special_tokens = ["[PAD]", "[UNK]", "[CLS]", "[SEP]", "[MASK]"]

    tokenizer = Tokenizer(BPE(unk_token="[UNK]"))
    tokenizer.pre_tokenizer = Whitespace()

    trainer = BpeTrainer(
        vocab_size=vocab_size,
        special_tokens=special_tokens,
        min_frequency=2  # ignore extremely rare pairs
    )

    tokenizer.train(
        files=[str(f) for f in corpus_files],
        trainer=trainer
    )

    # Optional: add simple post-processor to mimic [CLS] / [SEP] structure
    tokenizer.post_processor = TemplateProcessing(
        single="[CLS] $A [SEP]",
        pair="[CLS] $A [SEP] $B:1 [SEP]:1",
        special_tokens=[
            ("[CLS]", tokenizer.token_to_id("[CLS]")),
            ("[SEP]", tokenizer.token_to_id("[SEP]")),
        ],
    )

    return tokenizer

### Train Two Tokenizers

In [14]:
small_bpe = train_bpe_tokenizer([toy_corpus_file], vocab_size=200)
large_bpe = train_bpe_tokenizer([toy_corpus_file], vocab_size=2000)
print("Small vocab size:", small_bpe.get_vocab_size())
print("Large vocab size:", large_bpe.get_vocab_size())

Small vocab size: 98
Large vocab size: 98


### Compare Tokenization of Rare / Morphologically Rich Words

In [15]:
test_sentences = [
 "rareword ultrarareword ultra-rareword",
 "renew renews renewing renewal reset",
 "the lower newer lowest newest",
 "hyper-antidisestablishmentarian experiment",
 "multilingual tokenizers sometimes over-segment spanish palabras",
]
def show_encoding(tok, text):
 encoding = tok.encode(text)
 print("Text: ", text)
 print("Tokens:", encoding.tokens)
 print("IDs: ", encoding.ids)
 print("Len: ", len(encoding.tokens))
 print()
print("=== SMALL VOCAB BPE ===")
for s in test_sentences:
 show_encoding(small_bpe, s)
print("=== LARGE VOCAB BPE ===")
for s in test_sentences:
 show_encoding(large_bpe, s)

=== SMALL VOCAB BPE ===
Text:  rareword ultrarareword ultra-rareword
Tokens: ['[CLS]', 'rareword', 'ul', 't', 'rar', 'ar', 'eword', 'ul', 't', 'r', 'a', '-', 'rareword', '[SEP]']
IDs:  [2, 77, 91, 39, 71, 48, 74, 91, 39, 37, 20, 6, 77, 3]
Len:  14

Text:  renew renews renewing renewal reset
Tokens: ['[CLS]', 'renew', 'renew', 's', 'renew', 'ing', 'renew', 'al', 'r', 'es', 'e', 't', '[SEP]']
IDs:  [2, 66, 66, 38, 66, 95, 66, 81, 37, 54, 24, 39, 3]
Len:  13

Text:  the lower newer lowest newest
Tokens: ['[CLS]', 'th', 'e', 'low', 'er', 'new', 'er', 'low', 'est', 'new', 'est', '[SEP]']
IDs:  [2, 61, 24, 75, 47, 57, 47, 75, 94, 57, 94, 3]
Len:  12

Text:  hyper-antidisestablishmentarian experiment
Tokens: ['[CLS]', 'h', 'y', 'p', 'er', '-', 'a', 'n', 't', 'i', 'd', 'is', 'est', 'a', 'b', 'l', 'is', 'h', 'm', 'en', 't', 'ar', 'i', 'a', 'n', 'e', '[UNK]', 'p', 'er', 'i', 'm', 'en', 't', '[SEP]']
IDs:  [2, 27, 42, 35, 47, 6, 20, 33, 39, 28, 23, 68, 94, 20, 21, 31, 68, 27, 32, 45, 39, 48, 28, 

In [16]:
def avg_token_length(tok, sentences):
    lengths = []
    for s in sentences:
        lengths.append(len(tok.encode(s).tokens))
    return sum(lengths) / len(lengths), lengths
    
small_avg, small_lens = avg_token_length(small_bpe, test_sentences)
large_avg, large_lens = avg_token_length(large_bpe, test_sentences)

In [17]:
print("Token counts small vocab:", small_lens, "avg =", small_avg)
print("Token counts large vocab:", large_lens, "avg =", large_avg)

Token counts small vocab: [14, 13, 12, 34, 48] avg = 24.2
Token counts large vocab: [14, 13, 12, 34, 48] avg = 24.2


### Compare to a Pretrained Tokenizer

In [18]:
!pip install transformers


[notice] A new release of pip is available: 26.0 -> 26.0.1
[notice] To update, run: python.exe -m pip install --upgrade pip


In [19]:
from transformers import AutoTokenizer
gpt_like_tok = AutoTokenizer.from_pretrained("gpt2")
for s in test_sentences:
    toks = gpt_like_tok.tokenize(s)
    print("GPT-2:", s)
    print("Tokens:", toks)
    print("Len:", len(toks))
    print()

GPT-2: rareword ultrarareword ultra-rareword
Tokens: ['ra', 'rew', 'ord', 'Ġultr', 'ar', 'are', 'word', 'Ġultra', '-', 'ra', 'rew', 'ord']
Len: 12

GPT-2: renew renews renewing renewal reset
Tokens: ['ren', 'ew', 'Ġrenew', 's', 'Ġrenew', 'ing', 'Ġrenewal', 'Ġreset']
Len: 8

GPT-2: the lower newer lowest newest
Tokens: ['the', 'Ġlower', 'Ġnewer', 'Ġlowest', 'Ġnewest']
Len: 5

GPT-2: hyper-antidisestablishmentarian experiment
Tokens: ['hyper', '-', 'ant', 'idis', 'establishment', 'arian', 'Ġexperiment']
Len: 7

GPT-2: multilingual tokenizers sometimes over-segment spanish palabras
Tokens: ['mult', 'ilingual', 'Ġtoken', 'izers', 'Ġsometimes', 'Ġover', '-', 'se', 'gment', 'Ġsp', 'anish', 'Ġpal', 'ab', 'ras']
Len: 14

